# Q7. Monthly unit price of the five highest-value materials

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** How has the average unit price of the five highest-value materials changed month by
month (fixed-base and chain indices)?

Material = **product family** (see Q4). SAP material numbers are created per student, so a single
code rarely appears in more than one month.

- **Average unit price** in month *t* = Σ Net PO value / Σ order quantity (quantity-weighted). Only
  items with quantity > 0 and value > 0 in the family's main order unit are used.
- **Fixed-base index** = P_t / P_base × 100, where the base is the family's first month with purchases.
- **Chain index** = P_t / P_previous × 100, where "previous" is the previous month *with purchases*
  (empty months are skipped).

In [ ]:
w["family"] = product_family(w)
top5 = (w[w["family"] != "Other"].groupby("family")["Net Order Value"].sum()
          .sort_values(ascending=False).head(5))
display(top5.to_frame("net").style.format("{:,.0f}"))

main_unit = w.groupby("family")["Order Unit"].agg(lambda s: s.mode().iat[0])
p = w[w["family"].isin(top5.index) & (w["Order Quantity"] > 0) & (w["Net Order Value"] > 0)
      & (w["Order Unit"] == w["family"].map(main_unit))]
print("Main order unit:", main_unit[top5.index].to_dict())

In [ ]:
months = pd.period_range("2025-10", "2026-09", freq="M")
price = (p.groupby(["family", "month"]).agg(net=("Net Order Value", "sum"), qty=("Order Quantity", "sum"),
                                            items=("Item", "size"))
           .assign(price=lambda d: d.net / d.qty).reset_index())
price["month"] = pd.PeriodIndex(price["month"], freq="M")
price = price.sort_values(["family", "month"])
price["fixed_base"] = price.groupby("family")["price"].transform(lambda s: s / s.iat[0] * 100)
price["chain"] = price.groupby("family")["price"].transform(lambda s: s / s.shift() * 100)
price.style.format({"net": "{:,.0f}", "qty": "{:,.0f}", "price": "{:,.2f}",
                    "fixed_base": "{:.1f}", "chain": "{:.1f}"}, na_rep="–").hide(axis="index")

In [ ]:
idx = price.pivot(index="month", columns="family", values="fixed_base").reindex(months)[top5.index]
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(months))
for i, fam in enumerate(idx.columns):
    s = idx[fam]
    ok = s.notna().to_numpy()
    ax.plot(x[ok], s[ok], color=SERIES[i], ls=":", lw=1.2)          # bridges empty months
    ax.plot(x, s, color=SERIES[i], marker="o", mec="#fcfcfb", mew=1.5, label=fam)
ax.axhline(100, color=INK_2, lw=1)
ax.set_xticks(x, [m.strftime("%b\n%Y") if m.month in (1, 10) else m.strftime("%b") for m in months])
ax.set_xlim(-0.5, len(x) - 0.5); ax.set_yscale("log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set(title="Fixed-base price index (first month with purchases = 100, log scale)",
       ylabel="Index", xlabel="Dotted = months with no purchases")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
save(fig, "q7_fixed_base_index")

In [ ]:
fig, axes = plt.subplots(1, len(top5), figsize=(15, 3), sharey=False)
for i, (ax, fam) in enumerate(zip(axes, top5.index)):
    s = price[price["family"] == fam]
    ax.bar(s["month"].dt.strftime("%b %y"), s["price"], color=SERIES[i], width=0.6)
    ax.set_title(f"{fam} ({main_unit[fam]})", fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.tick_params(axis="x", rotation=90, labelsize=8); ax.grid(axis="x", visible=False)
fig.suptitle("Average unit price per month (USD, quantity-weighted), months with purchases only",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "q7_unit_price")

In [ ]:
# Price spread within the period: how much do suppliers differ for the same product?
(p.assign(unit_price=p["Net Order Value"] / p["Order Quantity"])
  .groupby("family")["unit_price"].describe(percentiles=[.25, .5, .75])
  .loc[top5.index].style.format("{:,.2f}"))

## Findings for Antonio

**Finding.** The five highest-value product families are chain lock, spelt flour, water bottle, touring bike and water bottle cage. Only **chain locks** have a usable price series: 7 months of purchases at **$32.0–35.0 per unit** (fixed-base index 98.5–107.7, chain index 94.4–104.4). Their price is stable, and the median unit price is $32. The other families jump by orders of magnitude between months. Water bottles go $9.5 → $51 → $10. Spelt flour goes from $13/kg to $288/kg, and touring bikes from $15 to $438 each. These come from a few exercise items with made-up prices, not from real price changes. Water bottle cages were bought in a single month ($9 each).

**Decision it supports.** For chain locks, the main spend item, a fixed-price long-term contract around $32–33 is supported by the data. For the other families the data can't support any price decision.

**Limitation / extra data.** Price indices need regular purchases of the *same* item. Here many months are empty, and material numbers differ per student, so we grouped by product family. Antonio would need contract and info-record prices, along with market price indices, to track price development properly.